# Demo — Synthetic 3D Line Reconstruction

This notebook walks through a complete **stereo (2-view) and multiview
(N-view) 3D line reconstruction** pipeline using a synthetic scene with
known ground truth.  The GT projection pipeline includes **depth-buffer
occlusion** so that only lines truly visible from each camera are used.

| Strategy | Description |
|----------|-------------|
| **GT Stereo** | Ground truth 2D lines + correspondences, stereo triangulation |
| **GT Multiview** | Ground truth 2D lines + tracks, N-view SVD triangulation |
| **LSD + GT-Assisted** | LSD-detected lines, oracle matching via GT parents |
| **LSD + Descriptor** | LSD-detected lines, LBD descriptor matching (fully automatic) |

**Prerequisites:**
- Build all bindings: `bazel build //libs/...`
- Kernel: `.venv` Python interpreter

**Sections:**
1. Setup & Imports
2. Build Scene & Camera Rig
3. Ground Truth Projection (with occlusion)
4. Wireframe Rendering
5. Textured Rendering & LSD Detection
6. 3D Reconstruction — GT Stereo
7. 3D Reconstruction — GT Multiview (N-View)
8. LSD Reconstruction — GT-Assisted Matching
9. LSD Reconstruction — Descriptor-Based Matching
10. Side-by-Side Comparison
11. Rerun Visualization

---
## 1. Setup & Imports

In [ ]:
import sys
import pathlib
import importlib

# Find workspace root (parent of examples/)
workspace = pathlib.Path.cwd()
while not (workspace / "MODULE.bazel").exists():
    if workspace == workspace.parent:
        raise RuntimeError("Cannot find LineExtraction workspace root (MODULE.bazel)")
    workspace = workspace.parent

# Add Bazel output paths for native extension modules
for lib in ["imgproc", "edge", "geometry", "eval", "lsd", "lfd", "algorithm"]:
    p = workspace / f"bazel-bin/libs/{lib}/python"
    if p.exists():
        sys.path.insert(0, str(p))

# Add lsfm package
sys.path.insert(0, str(workspace / "python"))

import numpy as np
import le_edge
import le_geometry
import le_lfd
import le_lsd

# Force-reload synthetic subpackage so source changes take effect
for _mod_name in sorted(k for k in sys.modules if k.startswith("lsfm.synthetic")):
    importlib.reload(sys.modules[_mod_name])

from lsfm.synthetic import (
    build_church_scene,
    CameraRig,
    make_stereo_camera,
    SceneGroundTruth,
    render_wireframe,
    render_all_views,
    render_textured,
    render_all_textured_views,
    oracle_tracks,
    evaluate_reconstruction,
    segments_to_array,
    segments3d_to_array,
)

print("All modules loaded successfully.")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


def show_images(images, titles=None, figsize=None, cols=4):
    """Display a list of grayscale or RGB images in a grid."""
    n = len(images)
    rows = (n + cols - 1) // cols
    if figsize is None:
        figsize = (4 * cols, 3.5 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = [axes]
    else:
        axes = axes.flat if hasattr(axes, "flat") else [axes]
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if img.ndim == 2:
                ax.imshow(img, cmap="gray", vmin=0, vmax=255)
            else:
                ax.imshow(img)
            if titles:
                ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def overlay_segments(img, segments, color=(255, 100, 50), thickness=1):
    """Draw 2D line segments on a grayscale or RGB image, returning RGB."""
    if img.ndim == 2:
        rgb = np.stack([img, img, img], axis=-1).copy()
    else:
        rgb = img.copy()
    for seg in segments:
        ep = seg.end_points()
        pt1 = (int(round(ep[0])), int(round(ep[1])))
        pt2 = (int(round(ep[2])), int(round(ep[3])))
        _draw_line(rgb, pt1, pt2, color, thickness)
    return rgb


def _draw_line(img, pt1, pt2, color, thickness):
    """Draw an anti-aliased line using numpy (avoids cv2 import)."""
    h, w = img.shape[:2]
    x0, y0 = pt1
    x1, y1 = pt2
    n_steps = max(abs(x1 - x0), abs(y1 - y0), 1)
    for t in np.linspace(0, 1, n_steps + 1):
        x = int(round(x0 + t * (x1 - x0)))
        y = int(round(y0 + t * (y1 - y0)))
        r = thickness // 2
        y_lo, y_hi = max(0, y - r), min(h, y + r + 1)
        x_lo, x_hi = max(0, x - r), min(w, x + r + 1)
        img[y_lo:y_hi, x_lo:x_hi] = color


def show_overlay(img, gt_segs, det_segs, title=""):
    """Show GT (green) vs detected (red) segments overlaid on an image."""
    rgb = overlay_segments(img, gt_segs, color=(0, 200, 0), thickness=2)
    rgb = overlay_segments(rgb, det_segs, color=(255, 80, 50), thickness=1)
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.title(title, fontsize=11)
    plt.axis("off")
    plt.show()


def to_grayscale(img):
    """Convert an RGB image to grayscale using luminance weights."""
    if img.ndim == 2:
        return img
    return (0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]).astype(np.uint8)


def to_f64(seg):
    """Convert a float LineSegment to LineSegment_f64."""
    if isinstance(seg, le_geometry.LineSegment_f64):
        return seg
    sp, ep = seg.start_point(), seg.end_point()
    return le_geometry.LineSegment_f64.from_endpoints(sp[0], sp[1], ep[0], ep[1])


print("Helpers defined.")

---
## 2. Build Scene & Camera Rig

We use the preset **church scene** (body + roof + tower + steeple) and an
**orbital camera rig** that places cameras evenly around the scene.

In [ ]:
# Build a church scene from primitives
scene = build_church_scene()
gt_lines_3d = scene.ground_truth_lines()

bb_min, bb_max = scene.bounding_box()
center = scene.center()

print(f"Scene: {len(scene.entries)} primitives")
for prim, pose in scene.entries:
    print(f"  - {prim.name}: {len(prim.edge_indices())} edges")
print(f"Ground truth 3D lines: {len(gt_lines_3d)} (deduplicated)")
print(f"Bounding box: [{bb_min}] to [{bb_max}]")
print(f"Center: {center}")

In [ ]:
N_CAMERAS = 8

rig = CameraRig.orbital(
    n_cameras=N_CAMERAS,
    target=tuple(center),
    radius=12.0,
    height=3.0,
    focal_length=500.0,
    image_width=640,
    image_height=480,
)

print(f"Camera rig: {len(rig.cameras)} cameras")
print(f"Image size: {int(rig.cameras[0].width)} x {int(rig.cameras[0].height)}")
print(f"Stereo pairs: {list(rig.pairs())}")

---
## 3. Ground Truth Projection (with Occlusion)

Project all 3D GT lines into every camera view.  The pipeline filters
lines behind the camera, performs **depth-buffer occlusion** to remove
lines hidden behind other primitives, projects to 2D via Pluecker
coordinates, and clips to the image frustum.

In [ ]:
gt = SceneGroundTruth(scene, rig, occlusion=True)

print(f"Views: {gt.n_views}")
for v in range(gt.n_views):
    segs = gt.segments_2d(v)
    parents = gt.parent_indices(v)
    print(f"  View {v}: {len(segs)} visible segments (from {len(set(parents))} unique 3D lines)")

# Show correspondences for first pair
corr_01 = gt.correspondences(0, 1)
print(f"\nCorrespondences (view 0 <-> view 1): {len(corr_01)} pairs")

---
## 4. Wireframe Rendering

Render clean wireframe images (black lines on white background).
These are ideal for testing the pipeline but challenging for
descriptor-based matching due to lack of texture.

In [ ]:
wireframes = render_all_views(gt, thickness=2)
show_images(wireframes, titles=[f"View {i}" for i in range(len(wireframes))])

---
## 5. Textured Rendering & LSD Detection

Re-render the scene with **procedural textures** and run LSD on the
textured images.  The textured renderings provide the image context
needed for descriptor-based line matching in later sections.

In [ ]:
# Render textured images WITHOUT GT edges (clean images for LSD)
textured = render_all_textured_views(scene, gt, draw_edges=False)

# Show with GT edges overlaid for reference
textured_display = render_all_textured_views(scene, gt, thickness=2, draw_edges=True)
show_images(textured_display, titles=[f"Textured view {i}" for i in range(len(textured_display))])

In [ ]:
MIN_LENGTH = 20  # minimum segment length in pixels
TH_RATIO = 0.33  # ratio of th_low to th_high
OTSU_SCALE = 0.28  # scale factor for Otsu threshold (0.5 = half of computed value)

# Run LSD on textured images with Otsu threshold and min_pix
detected_tex = []
for i, img in enumerate(textured):
    gray = to_grayscale(img)
    # Compute gradient magnitude for Otsu threshold estimation
    es = le_edge.EdgeSourceSobel()
    es.process(gray)
    mag_max = es.magnitude_max()
    otsu_th = le_edge.ThresholdOtsu(mag_max).process(es.magnitude())
    th_high = otsu_th / mag_max * OTSU_SCALE # normalize to 0-1
    th_low = th_high * TH_RATIO

    det = le_lsd.LsdCC(th_low=th_low, th_high=th_high, min_pix=MIN_LENGTH)
    det.detect(gray)
    segs = det.line_segments()
    detected_tex.append(segs)
    print(f"View {i}: {len(segs)} segments detected "
          f"(GT: {len(gt.segments_2d(i))}, Otsu th={th_high:.4f})")

overlays = [overlay_segments(img, segs, color=(50, 100, 255), thickness=2)
            for img, segs in zip(textured, detected_tex)]
show_images(overlays, titles=[f"View {i}: LSD on textured ({len(detected_tex[i])} segs)"
                              for i in range(len(overlays))])

---
## 6. 3D Reconstruction — GT Stereo

Create a **dedicated stereo pair** by placing a second camera next to one
of the orbital cameras (camera 1) with the same orientation but shifted
by a small baseline along the local X-axis.  This mimics a real stereo
rig — two sensors side-by-side looking in the same direction.

In [ ]:
# Pick an orbital camera as the left eye, create a stereo partner as the right eye
STEREO_REF = 1  # orbital camera index used as left eye
STEREO_BASELINE = 0.5  # meters

cam_l = rig.cameras[STEREO_REF]
cam_r = make_stereo_camera(cam_l, baseline=STEREO_BASELINE)

# Build a 2-camera rig and compute GT projections for the stereo pair
stereo_rig = CameraRig([cam_l, cam_r])
stereo_gt = SceneGroundTruth(scene, stereo_rig, occlusion=True)

print(f"Stereo pair: orbital cam {STEREO_REF} + partner (baseline={STEREO_BASELINE})")
print(f"  Left view:  {len(stereo_gt.segments_2d(0))} GT segments")
print(f"  Right view: {len(stereo_gt.segments_2d(1))} GT segments")

# GT correspondences between left and right
gt_corr = stereo_gt.correspondences(0, 1)
print(f"  Correspondences: {len(gt_corr)} pairs")

# Triangulate
stereo = le_geometry.Stereo_f64(cam_l, cam_r)
matched_l = [stereo_gt.segments_2d(0)[il] for il, _ in gt_corr]
matched_r = [stereo_gt.segments_2d(1)[ir] for _, ir in gt_corr]
segs3d_gt_stereo = stereo.triangulate_segments(matched_l, matched_r)

print(f"  3D segments:  {len(segs3d_gt_stereo)}")
print(f"  Ground truth: {len(gt.gt_lines_3d)}")

recon_gt_stereo = evaluate_reconstruction(segs3d_gt_stereo, gt.gt_lines_3d)
print(f"\nGT Stereo reconstruction:")
print(f"  Distance (median): {recon_gt_stereo.distance_median:.4f}")
print(f"  Angle (median):    {recon_gt_stereo.angle_median:.2f} deg")
print(f"  Length ratio:      {recon_gt_stereo.length_ratio_mean:.3f}")
print(f"  Matched: {recon_gt_stereo.n_matched}/{recon_gt_stereo.n_gt} GT lines")

---
## 7. 3D Reconstruction — GT Multiview (N-View)

Use all **N cameras** simultaneously to triangulate 3D lines.  The
`MultiviewStereo` class constructs interpretation planes from each 2D
observation and recovers the 3D line through SVD-based least squares.

Tracks are built directly from **GT parent correspondences** — each
3D parent line that is visible in two or more views produces one track.

In [ ]:
# Build tracks directly from GT parent indices (no detection/matching)
tracks = gt.gt_tracks()
print(f"GT tracks: {len(tracks)} (spanning {N_CAMERAS} views)")
for t in tracks[:5]:
    views = sorted({v for v, _ in t})
    print(f"  Track with {len(t)} observations in views {views}")

# Prepare segment observations for the multiview triangulator
mv = le_geometry.MultiviewStereo_f64(rig.cameras)

track_observations = []
for track in tracks:
    obs = [(v_idx, gt.segments_2d(v_idx)[seg_idx]) for v_idx, seg_idx in track]
    track_observations.append(obs)

# Triangulate all tracks
segs3d_gt_mv = mv.triangulate_segments(track_observations)

# Filter empty results (degenerate tracks)
segs3d_gt_mv = [s for s in segs3d_gt_mv if not s.empty()]

print(f"\nMultiview triangulation results:")
print(f"  Input tracks:    {len(tracks)}")
print(f"  3D segments:     {len(segs3d_gt_mv)}")
print(f"  Ground truth:    {len(gt.gt_lines_3d)}")

recon_gt_mv = evaluate_reconstruction(segs3d_gt_mv, gt.gt_lines_3d)
print(f"\nGT Multiview reconstruction:")
print(f"  Distance (median): {recon_gt_mv.distance_median:.4f}")
print(f"  Angle (median):    {recon_gt_mv.angle_median:.2f} deg")
print(f"  Length ratio:      {recon_gt_mv.length_ratio_mean:.3f}")
print(f"  Matched: {recon_gt_mv.n_matched}/{recon_gt_mv.n_gt} GT lines")

---
## 8. LSD Reconstruction — GT-Assisted Matching

Use **LSD-detected lines** from the textured images, but match them
across views using **ground truth knowledge**.  Each detected segment is
assigned to the nearest GT segment (greedy midpoint + angle cost), then
cross-view correspondences are derived from the shared 3D parent index.

This isolates **detection noise** from matching noise — we see how well
the triangulator handles real (imperfect) 2D segments when given correct
correspondences.

In [ ]:
# Build multiview tracks from LSD-on-textured detections using GT parent knowledge
lsd_tracks_gt = oracle_tracks(detected_tex, gt)

print(f"Oracle tracks from LSD detections: {len(lsd_tracks_gt)}")
for t in lsd_tracks_gt[:5]:
    views = sorted({v for v, _ in t})
    print(f"  Track with {len(t)} observations in views {views}")

# Triangulate using MultiviewStereo with all cameras
mv = le_geometry.MultiviewStereo_f64(rig.cameras)

track_obs_lsd_gt = []
for track in lsd_tracks_gt:
    obs = [(v_idx, to_f64(detected_tex[v_idx][seg_idx])) for v_idx, seg_idx in track]
    track_obs_lsd_gt.append(obs)

segs3d_lsd_gt = mv.triangulate_segments(track_obs_lsd_gt)
segs3d_lsd_gt = [s for s in segs3d_lsd_gt if not s.empty()]

# Filter by max plausible length (scene bounding box diagonal)
bb_diag = float(np.linalg.norm(bb_max - bb_min))
MAX_LENGTH = bb_diag * 1.5
segs3d_lsd_gt = [s for s in segs3d_lsd_gt if s.length <= MAX_LENGTH]

print(f"\nLSD + GT-Assisted Multiview reconstruction:")
print(f"  Input tracks:    {len(lsd_tracks_gt)}")
print(f"  3D segments:     {len(segs3d_lsd_gt)}")
print(f"  Ground truth:    {len(gt.gt_lines_3d)}")

recon_lsd_gt = evaluate_reconstruction(segs3d_lsd_gt, gt.gt_lines_3d)
print(f"\n  Distance (median): {recon_lsd_gt.distance_median:.4f}")
print(f"  Angle (median):    {recon_lsd_gt.angle_median:.2f} deg")
print(f"  Length ratio:      {recon_lsd_gt.length_ratio_mean:.3f}")
print(f"  Matched: {recon_lsd_gt.n_matched}/{recon_lsd_gt.n_gt} GT lines")

---
## 9. LSD Reconstruction — Descriptor-Based Matching

Use **OpenCV Binary LBD** descriptors to match LSD-detected lines across
views *without* any ground truth knowledge.  Matching pipeline:

1. Compute binary LBD descriptors for each view's LSD segments.
2. For each view pair, run KNN (k=2) matching + **Lowe's ratio test**
   to reject ambiguous matches.
3. Enforce **mutual consistency** (both directions must agree).
4. Build multiview tracks via transitive closure of pairwise matches.
5. Triangulate with `MultiviewStereo`.
6. **Filter outliers** by removing 3D segments with implausible length.

In [ ]:
from collections import defaultdict

# --- 1. Compute binary LBD descriptors for each view ---
grays = [to_grayscale(img) for img in textured]

descriptors_per_view = []
for i, gray in enumerate(grays):
    fdc = le_lfd.FdcOpenCVLBD(gray)
    descs = fdc.create_list(detected_tex[i])
    descriptors_per_view.append(descs)
    print(f"View {i}: {len(descs)} descriptors for {len(detected_tex[i])} segments")


# --- 2 & 3. Pairwise matching with ratio test + mutual consistency ---
RATIO_THRESH = 0.75  # Lowe's ratio test

def match_pair(descs_i, descs_j):
    """KNN match + ratio test + mutual consistency."""
    if len(descs_i) < 2 or len(descs_j) < 2:
        return []
    # Forward: i -> j
    matcher_fwd = le_lfd.BruteForceOpenCVLBD()
    matcher_fwd.train(descs_i, descs_j)
    knn_fwd = matcher_fwd.knn(2)
    # Group by query
    fwd_by_q = defaultdict(list)
    for m in knn_fwd:
        fwd_by_q[m.query_idx].append(m)
    # Ratio test: keep only if best << second best
    fwd_good = {}
    for qi, ms in fwd_by_q.items():
        ms.sort(key=lambda m: m.distance)
        if len(ms) >= 2 and ms[1].distance > 0:
            if ms[0].distance / ms[1].distance < RATIO_THRESH:
                fwd_good[qi] = ms[0].match_idx

    # Backward: j -> i
    matcher_bwd = le_lfd.BruteForceOpenCVLBD()
    matcher_bwd.train(descs_j, descs_i)
    knn_bwd = matcher_bwd.knn(2)
    bwd_by_q = defaultdict(list)
    for m in knn_bwd:
        bwd_by_q[m.query_idx].append(m)
    bwd_good = {}
    for qi, ms in bwd_by_q.items():
        ms.sort(key=lambda m: m.distance)
        if len(ms) >= 2 and ms[1].distance > 0:
            if ms[0].distance / ms[1].distance < RATIO_THRESH:
                bwd_good[qi] = ms[0].match_idx

    # Mutual consistency
    matches = []
    for i_idx, j_idx in fwd_good.items():
        if bwd_good.get(j_idx) == i_idx:
            matches.append((i_idx, j_idx))
    return matches


pairwise_matches = {}
for vi in range(N_CAMERAS):
    for vj in range(vi + 1, N_CAMERAS):
        matches = match_pair(descriptors_per_view[vi], descriptors_per_view[vj])
        pairwise_matches[(vi, vj)] = matches
        if matches:
            print(f"  Views ({vi},{vj}): {len(matches)} mutual matches")


# --- 4. Build multiview tracks via union-find ---
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


uf = UnionFind()
for (vi, vj), matches in pairwise_matches.items():
    for si, sj in matches:
        uf.union((vi, si), (vj, sj))

# Group into tracks
track_groups = defaultdict(list)
all_obs = set()
for (vi, vj), matches in pairwise_matches.items():
    for si, sj in matches:
        all_obs.add((vi, si))
        all_obs.add((vj, sj))

for obs in all_obs:
    root = uf.find(obs)
    track_groups[root].append(obs)

# Keep tracks with >=2 different views
desc_tracks = []
for obs_list in track_groups.values():
    views_seen = {v for v, _ in obs_list}
    if len(views_seen) >= 2:
        desc_tracks.append(obs_list)

print(f"\nDescriptor-based tracks: {len(desc_tracks)}")
for t in desc_tracks[:5]:
    views = sorted({v for v, _ in t})
    print(f"  Track with {len(t)} observations in views {views}")

In [ ]:
# --- 5. Triangulate descriptor-based tracks ---
track_obs_desc = []
for track in desc_tracks:
    # Deduplicate: keep one observation per view (first seen)
    seen = {}
    for v_idx, seg_idx in track:
        if v_idx not in seen:
            seen[v_idx] = seg_idx
    obs = [(v, to_f64(detected_tex[v][si])) for v, si in seen.items()]
    if len(obs) >= 2:
        track_obs_desc.append(obs)

segs3d_desc = mv.triangulate_segments(track_obs_desc)
segs3d_desc = [s for s in segs3d_desc if not s.empty()]

# --- 6. Filter outliers by plausible length ---
bb_diag = float(np.linalg.norm(bb_max - bb_min))
MAX_LENGTH = bb_diag * 1.5
n_before = len(segs3d_desc)
segs3d_desc = [s for s in segs3d_desc if s.length <= MAX_LENGTH]
print(f"Length filter: {n_before} -> {len(segs3d_desc)}  (removed {n_before - len(segs3d_desc)} outliers, threshold={MAX_LENGTH:.2f})")

# --- 7. Evaluate ---
print(f"\nLSD + Descriptor-Based Multiview reconstruction:")
print(f"  Input tracks:    {len(desc_tracks)}")
print(f"  3D segments:     {len(segs3d_desc)}")
print(f"  Ground truth:    {len(gt.gt_lines_3d)}")

recon_desc = evaluate_reconstruction(segs3d_desc, gt.gt_lines_3d)
print(f"\n  Distance (median): {recon_desc.distance_median:.4f}")
print(f"  Angle (median):    {recon_desc.angle_median:.2f} deg")
print(f"  Length ratio:      {recon_desc.length_ratio_mean:.3f}")
print(f"  Matched: {recon_desc.n_matched}/{recon_desc.n_gt} GT lines")

---
## 10. Side-by-Side Comparison

Compare all four reconstruction strategies.

In [ ]:
def fmt(val, decimals=3):
    """Format a numeric value for the comparison table."""
    if isinstance(val, int):
        return str(val)
    return f"{val:.{decimals}f}"


header = (
    f"| {'Metric':<22} | {'GT Stereo':>12} | {'GT Multiview':>14} "
    f"| {'LSD+GT':>12} | {'LSD+Desc':>12} |"
)
sep = f"|{'-'*24}|{'-'*14}|{'-'*16}|{'-'*14}|{'-'*14}|"

rows = [
    ("3D Dist (median)",
     recon_gt_stereo.distance_median, recon_gt_mv.distance_median,
     recon_lsd_gt.distance_median, recon_desc.distance_median),
    ("3D Angle (median)",
     recon_gt_stereo.angle_median, recon_gt_mv.angle_median,
     recon_lsd_gt.angle_median, recon_desc.angle_median),
    ("Length Ratio",
     recon_gt_stereo.length_ratio_mean, recon_gt_mv.length_ratio_mean,
     recon_lsd_gt.length_ratio_mean, recon_desc.length_ratio_mean),
    ("3D Lines",
     len(segs3d_gt_stereo), len(segs3d_gt_mv),
     len(segs3d_lsd_gt), len(segs3d_desc)),
    ("GT Matched",
     recon_gt_stereo.n_matched, recon_gt_mv.n_matched,
     recon_lsd_gt.n_matched, recon_desc.n_matched),
    ("GT Total",
     recon_gt_stereo.n_gt, recon_gt_mv.n_gt,
     recon_lsd_gt.n_gt, recon_desc.n_gt),
]

print(header)
print(sep)
for metric, v1, v2, v3, v4 in rows:
    d = 4 if "Dist" in metric else 2 if "Angle" in metric else 3
    print(
        f"| {metric:<22} | {fmt(v1, d):>12} | {fmt(v2, d):>14} "
        f"| {fmt(v3, d):>12} | {fmt(v4, d):>12} |"
    )

---
## 11. Rerun Visualization

Log the full pipeline to **Rerun** for interactive 3D exploration.

**Timeline steps:**

| Step | Content |
|------|---------|
| 0 | 3D scene geometry (GT lines, green) + camera frustums |
| 1 | GT Stereo reconstruction (orange) vs GT (green) |
| 2 | GT Multiview reconstruction (magenta) vs GT (green) |
| 3 | LSD + GT-Assisted reconstruction (cyan) |
| 4 | LSD + Descriptor-Based reconstruction (yellow) |

In [ ]:
try:
    import rerun as rr
    HAS_RERUN = True
except ImportError:
    HAS_RERUN = False
    print("Rerun not installed — skipping visualization.")
    print("Install with: pip install rerun-sdk")

In [ ]:
if HAS_RERUN:
    import rerun.blueprint as rrb
    from lsfm.synthetic.rerun_viz import (
        log_scene,
        log_cameras,
        log_reconstruction,
        reconstruction_blueprint,
    )

    rr.init("synthetic_3d_recon")

    blueprint = reconstruction_blueprint(min(N_CAMERAS, 4))
    if blueprint is not None:
        rr.send_blueprint(blueprint)

    # Step 0: Scene geometry (GT 3D lines in green) + cameras
    rr.set_time("step", sequence=0)
    log_scene(scene)
    log_cameras(
        rig,
        highlight={
            STEREO_REF: ((255, 100, 50), f"Stereo L (cam {STEREO_REF})"),
        },
    )
    log_cameras(
        stereo_rig,
        entity_path="world/stereo_pair",
        highlight={
            1: ((255, 100, 50), "Stereo R"),
        },
    )

    # Step 1: GT Stereo reconstruction (orange)
    rr.set_time("step", sequence=1)
    log_reconstruction(segs3d_gt_stereo, entity_path="world/reconstruction/gt_stereo")

    # Step 2: GT Multiview reconstruction (magenta)
    rr.set_time("step", sequence=2)
    log_reconstruction(
        segs3d_gt_mv,
        entity_path="world/reconstruction/gt_multiview",
        color=(220, 50, 220),
    )

    # Step 3: LSD + GT-Assisted reconstruction (cyan)
    rr.set_time("step", sequence=3)
    log_reconstruction(
        segs3d_lsd_gt,
        entity_path="world/reconstruction/lsd_gt",
        color=(0, 200, 220),
    )

    # Step 4: LSD + Descriptor-Based reconstruction (yellow)
    rr.set_time("step", sequence=4)
    log_reconstruction(
        segs3d_desc,
        entity_path="world/reconstruction/lsd_descriptor",
        color=(230, 200, 50),
    )

    rr.notebook_show(width=1600, height=900)

    from IPython.display import display, HTML
    display(HTML(
        '<div style="display:flex;gap:24px;margin-top:6px;font-size:13px">'
        '<span><b style="color:#00c800">&#9644;</b> Ground Truth</span>'
        '<span><b style="color:#ff6432">&#9644;</b> GT Stereo (2-view)</span>'
        '<span><b style="color:#dc32dc">&#9644;</b> GT Multiview (N-view)</span>'
        '<span><b style="color:#00c8dc">&#9644;</b> LSD + GT-Assisted</span>'
        '<span><b style="color:#e6c832">&#9644;</b> LSD + Descriptor</span>'
        '</div>'
    ))

---

## Summary

Four reconstruction strategies are compared, ordered from perfect
information to fully automatic:

| Method | Matching | Detection | Key Insight |
|--------|----------|-----------|-------------|
| **GT Stereo** | GT correspondences | GT segments | Pure triangulation quality (2-view baseline) |
| **GT Multiview** | GT correspondences | GT segments | SVD plane intersection from N views — lower angular error |
| **LSD + GT-Assisted** | Oracle tracks (via GT parent lines) | LSD on textured images | Real detections, but matching bypasses descriptor noise |
| **LSD + Descriptor** | LBD descriptors + ratio test | LSD on textured images | Fully automatic — no GT knowledge at matching time |

**Key observations:**
- **GT Multiview** benefits from redundant observations and typically
  achieves lower angular error and better GT coverage than stereo.
- **LSD + GT-Assisted** shows how detection noise (segment splitting,
  endpoint jitter) degrades reconstruction even with perfect matching.
- **LSD + Descriptor** is fully automatic.  Quality depends on
  descriptor distinctiveness and the ratio-test threshold.
  Outlier filtering by 3D segment length is essential.
- Ground truth 3D lines (green) are logged at **step 0** in Rerun and
  remain visible in all subsequent steps for comparison.